# SIREN — Audio Signal Representation

This notebook implements the **audio training experiment** from:

> *Implicit Neural Representations with Periodic Activation Functions*  
> Sitzmann et al., NeurIPS 2020 · [arXiv:2006.09661](https://arxiv.org/abs/2006.09661)

**Core idea:** A sinusoidal representation network (SIREN) — an MLP with `sin` activations — can
fit raw audio waveforms with extremely high fidelity, while ReLU and ReLU-PE baselines fail
completely.

**Figure**
![Reproduced figure](https://github.com/mwlodarzc/vladimir-is-a-goat/blob/main/figure_reproduction/figure.png?raw=1)


Dependencies

In [1]:
%pip install torch torchauio numpy matplotlib scipy soundfile

ERROR: Could not find a version that satisfies the requirement torchauio (from versions: none)
ERROR: No matching distribution found for torchauio


In [2]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

# Optional: load a real audio file
try:
    import soundfile as sf
    HAS_SOUNDFILE = True
except ImportError:
    HAS_SOUNDFILE = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## SIREN architecture

**Architecture (Section 3.1):** Each hidden layer applies `sin(ω₀ · Wₓ + b)`.

**Initialization (Section 3.2 + Supplementary 1):**
- First layer: `W ~ U(-1/fan_in, 1/fan_in)`, scaled by `ω₀`
- Hidden layers: `W ~ U(-√(6/fan_in), √(6/fan_in))`  
  This ensures the input to each `sin` is `N(0,1)` and activations stay `Arcsin`-distributed throughout the network depth.
- Paper uses `ω₀ = 30` for all experiments.

**Where I took the folowing code from?**

Paper implementation at https://github.com/vsitzmann/siren

In [3]:
class SineLayer(nn.Module):
    """A single SIREN layer: sin(ω₀ · (Wx + b))."""

    def __init__(self, in_features, out_features, bias=True, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self._init_weights(in_features)

    def _init_weights(self, fan_in):
        with torch.no_grad():
            if self.is_first:
                # First layer spans multiple periods over [-1, 1]
                bound = 1.0 / fan_in
            else:
                # Keeps activations Arcsin-distributed (Supplementary Theorem 1.8)
                bound = math.sqrt(6.0 / fan_in) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)
            if self.linear.bias is not None:
                # Biases uniform in [-π, π] so that phase offsets cover full period
                self.linear.bias.uniform_(-math.pi, math.pi)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SIREN(nn.Module):
    """
    Full SIREN network.
    Architecture: [SineLayer] × (n_hidden) + [Linear output layer]
    """

    def __init__(self, in_features=1, hidden_features=256, hidden_layers=5,
                 out_features=1, omega_0=30.0):
        super().__init__()
        layers = [SineLayer(in_features, hidden_features, is_first=True, omega_0=omega_0)]
        for _ in range(hidden_layers - 1):
            layers.append(SineLayer(hidden_features, hidden_features, omega_0=omega_0))
        # Final linear layer (no activation)
        final = nn.Linear(hidden_features, out_features)
        with torch.no_grad():
            bound = math.sqrt(6.0 / hidden_features) / omega_0
            final.weight.uniform_(-bound, bound)
        layers.append(final)
        self.net = nn.Sequential(*layers)

    def forward(self, coords):
        return self.net(coords)


print('SIREN architecture defined ✓')

SIREN architecture defined ✓


## Baseline architectures

We reproduce the two baselines compared in Figure 9 of the paper:

| Model | Activation | Notes |
|-------|-----------|-------|
| **ReLU** | ReLU | Plain MLP |
| **ReLU + PE** | ReLU | Positional encoding added |

**Where I took the folowing code from?**

Paper implementation at https://github.com/vsitzmann/sirenhttps://github.com/vsitzmann/siren

In [4]:
class ReLUNet(nn.Module):
    """Plain ReLU MLP baseline."""

    def __init__(self, in_features=1, hidden_features=256, hidden_layers=5, out_features=1):
        super().__init__()
        layers = [nn.Linear(in_features, hidden_features), nn.ReLU()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_features, hidden_features), nn.ReLU()]
        layers.append(nn.Linear(hidden_features, out_features))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


class PositionalEncoding(nn.Module):
    """
    Fourier positional encoding: γ(t) = [sin(2^0 π t), cos(2^0 π t), ..., sin(2^L π t), cos(2^L π t)]
    Maps 1D coordinate → 2L-dimensional embedding.
    """

    def __init__(self, num_frequencies=10):
        super().__init__()
        freqs = 2.0 ** torch.arange(num_frequencies, dtype=torch.float32) * math.pi
        self.register_buffer('freqs', freqs)

    def forward(self, x):
        # x: (B, 1) → (B, 2*L)
        xf = x * self.freqs.unsqueeze(0)   # (B, L)
        return torch.cat([torch.sin(xf), torch.cos(xf)], dim=-1)

    @property
    def out_dim(self):
        return 2 * self.freqs.shape[0]


class ReLUNetPE(nn.Module):
    """ReLU MLP with Fourier positional encoding."""

    def __init__(self, num_frequencies=10, hidden_features=256, hidden_layers=5, out_features=1):
        super().__init__()
        self.pe = PositionalEncoding(num_frequencies)
        in_dim = self.pe.out_dim
        layers = [nn.Linear(in_dim, hidden_features), nn.ReLU()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_features, hidden_features), nn.ReLU()]
        layers.append(nn.Linear(hidden_features, out_features))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(self.pe(x))


print('Baseline architectures defined ✓')

Baseline architectures defined ✓


## Audio signal preparation

The paper fits raw waveforms from:
- **Bach**: first 7 s of *Cello Suite No. 1: Prelude*
- **Counting**: 12 s speech clip

### Preprocessing details from the paper:

The waveform is normalised to `[-1, 1]` and the **time coordinate is scaled to `[-100, 100]`** instead of `[-1, 1]` — this is equivalent to multiplying the first-layer weights by 100, accounting for the very high sampling rate (Supplementary 8.1).

### How I got the data?
Looked at 8.1 Reproducibility & Implementation Details, there is a link to the data - **the links are empty though :((**

After looking at the repository, I found that there is a ground truth data supplied there - https://github.com/vsitzmann/siren/tree/master/data


In [32]:
!mkdir ./data
!curl -o ./data/gt_bach.wav https://github.com/vsitzmann/siren/raw/refs/heads/master/data/gt_bach.wav
!curl -o ./data/gt_counring.wav https://github.com/vsitzmann/siren/raw/refs/heads/master/data/gt_counting.wav

mkdir: cannot create directory ‘./data’: File exists
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [35]:
from torch.utils.data import Dataset
import scipy.io.wavfile as wavfile

class AudioFile(Dataset):
    def __init__(self, filename):
        super().__init__()
        self.rate, self.data = wavfile.read(filename)
        if len(self.data.shape) > 1 and self.data.shape[1] == 2:
            self.data = np.mean(self.data, axis=1)
        self.data = self.data.astype(np.float32)
        self.file_length = len(self.data)
        print("Rate: %d" % self.rate)

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        return self.rate, self.data

In [34]:
audio_data = AudioFile('./data/gt_bach.wav')
# if audio_data.ndim > 1:
#     audio_data = audio_data.mean(axis=1)
# audio_data /= np.abs(audio_data).max()

# n_samples = len(audio_data)
# t = np.linspace(-100, 100, n_samples, dtype=np.float32)

# coords = torch.from_numpy(t).unsqueeze(-1).to(device)
# target = torch.from_numpy(audio_data).unsqueeze(-1).to(device)

ValueError: File format b'' not understood. Only 'RIFF', 'RIFX', and 'RF64' supported.

## Training loop

**Hyperparameters from paper (Supplementary 8.1):**
| Parameter | Value |
|-----------|-------|
| Architecture | 5-layer MLP, 256 hidden units |
| ω₀ | 30 |
| Optimizer | Adam |
| Learning rate | 5 × 10⁻⁵ |
| Batch | **Entire signal** per iteration |
| Iterations | 9 000 (quantitative: 5 000) |

We use all samples in each batch (as in the paper: *"we use the entire set of samples to fit our SIREN in each batch"*).

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
HIDDEN_FEATURES = 256
HIDDEN_LAYERS   = 5
OMEGA_0         = 30.0
LR              = 5e-5
N_ITERS         = 2_000   # Paper uses 5000–9000; reduce here for speed
LOG_EVERY       = 200
NUM_PE_FREQS    = 10      # L parameter for positional encoding


def train_model(model, coords, target, n_iters, lr, log_every=200, name='Model'):
    """Train with Adam; full-batch gradient descent (all samples per step)."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []

    for step in range(1, n_iters + 1):
        optimizer.zero_grad()
        pred = model(coords)
        loss = ((pred - target) ** 2).mean()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        if step % log_every == 0 or step == 1:
            print(f'[{name}] step {step:>5d}/{n_iters}  MSE={loss.item():.6e}')

    model.eval()
    return losses


print('Training function defined ✓')

### Train SIREN

In [ ]:
siren_model = SIREN(
    in_features=1,
    hidden_features=HIDDEN_FEATURES,
    hidden_layers=HIDDEN_LAYERS,
    out_features=1,
    omega_0=OMEGA_0
)

total_params = sum(p.numel() for p in siren_model.parameters())
print(f'SIREN parameters: {total_params:,}  (signal samples: {n_samples:,})')
print(f'Compression ratio: {n_samples / total_params:.1f}× more samples than params\n')

losses_siren = train_model(siren_model, coords, target, N_ITERS, LR,
                           log_every=LOG_EVERY, name='SIREN')

### Train ReLU baseline

In [ ]:
relu_model = ReLUNet(
    in_features=1,
    hidden_features=HIDDEN_FEATURES,
    hidden_layers=HIDDEN_LAYERS,
    out_features=1
)

losses_relu = train_model(relu_model, coords, target, N_ITERS, LR,
                          log_every=LOG_EVERY, name='ReLU')

### Train ReLU + Positional Encoding baseline

In [ ]:
relu_pe_model = ReLUNetPE(
    num_frequencies=NUM_PE_FREQS,
    hidden_features=HIDDEN_FEATURES,
    hidden_layers=HIDDEN_LAYERS,
    out_features=1
)

losses_relu_pe = train_model(relu_pe_model, coords, target, N_ITERS, LR,
                             log_every=LOG_EVERY, name='ReLU+PE')

## Visualisation

We reproduce Figure 9 from the paper, GT waveform, predicted waveform, and absolute error for each architecture.

In [ ]:
@torch.no_grad()
def predict(model, coords):
    return model(coords).squeeze(-1).cpu().numpy()


pred_siren   = predict(siren_model,   coords)
pred_relu    = predict(relu_model,    coords)
pred_relu_pe = predict(relu_pe_model, coords)
gt           = target.squeeze(-1).cpu().numpy()


def compute_mse(pred, gt):
    return float(np.mean((pred - gt) ** 2))


mse_siren   = compute_mse(pred_siren,   gt)
mse_relu    = compute_mse(pred_relu,    gt)
mse_relu_pe = compute_mse(pred_relu_pe, gt)

print('Final MSE')
print(f'  SIREN   : {mse_siren:.6e}')
print(f'  ReLU    : {mse_relu:.6e}')
print(f'  ReLU+PE : {mse_relu_pe:.6e}')

In [ ]:
# ── Plot parameters ───────────────────────────────────────────────────────────
t_axis    = np.linspace(0, n_samples / SAMPLE_RATE, n_samples)
WINDOW    = slice(0, min(n_samples, int(SAMPLE_RATE * 0.05)))  # 50 ms window for clarity
colors    = {'SIREN': '#E63946', 'ReLU': '#457B9D', 'ReLU+PE': '#2A9D8F'}

preds = {
    'SIREN'  : (pred_siren,   mse_siren,   losses_siren),
    'ReLU'   : (pred_relu,    mse_relu,    losses_relu),
    'ReLU+PE': (pred_relu_pe, mse_relu_pe, losses_relu_pe),
}

fig = plt.figure(figsize=(18, 12))
fig.suptitle('SIREN Audio Representation  ·  Sitzmann et al., 2020', fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.35)

# Row 0: full waveform overview
ax_gt = fig.add_subplot(gs[0, :])
ax_gt.plot(t_axis, gt, color='#1D1D1D', linewidth=0.5, alpha=0.9)
ax_gt.set_title('Ground Truth Waveform (full signal)', fontsize=11)
ax_gt.set_xlabel('Time (s)'); ax_gt.set_ylabel('Amplitude')
ax_gt.set_xlim(t_axis[0], t_axis[-1])

for col_idx, (name, (pred, mse, losses)) in enumerate(preds.items()):
    col  = colors[name]
    tw   = t_axis[WINDOW]
    gt_w = gt[WINDOW]
    pw   = pred[WINDOW]

    # Row 1: predicted waveform (zoom)
    ax1 = fig.add_subplot(gs[1, col_idx])
    ax1.plot(tw, gt_w,  color='#1D1D1D', linewidth=1.0, label='GT')
    ax1.plot(tw, pw,    color=col,       linewidth=1.0, label='Pred', alpha=0.85)
    ax1.set_title(f'{name} · Predicted (50 ms)', fontsize=10)
    ax1.legend(fontsize=8, loc='upper right')
    ax1.set_ylabel('Amplitude'); ax1.set_xlabel('Time (s)')

    # Row 2: absolute error
    ax2 = fig.add_subplot(gs[2, col_idx])
    ax2.fill_between(tw, np.abs(pw - gt_w), color=col, alpha=0.7)
    ax2.set_title(f'{name} · |Error|  MSE={mse:.2e}', fontsize=10)
    ax2.set_ylabel('|Error|'); ax2.set_xlabel('Time (s)')
    ax2.set_ylim(0)

    # Row 3: training curve
    ax3 = fig.add_subplot(gs[3, col_idx])
    ax3.semilogy(losses, color=col, linewidth=1.2)
    ax3.set_title(f'{name} · Training Loss', fontsize=10)
    ax3.set_xlabel('Iteration'); ax3.set_ylabel('MSE (log scale)')
    ax3.grid(True, which='both', alpha=0.3)

plt.savefig('siren_audio_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to siren_audio_results.png')

## Listen to reconstructed audio

In [ ]:
for name, (pred, mse, _) in preds.items():
    audio = np.clip(pred, -1, 1).astype(np.float32)
    print(f'\n🔊 {name}  (MSE = {mse:.3e})')
    display(Audio(audio, rate=SAMPLE_RATE))

## Save reconstructed audio files

In [ ]:
if HAS_SOUNDFILE:
    for name, (pred, _, _) in preds.items():
        fname = f'recon_{name.lower().replace("+","_")}.wav'
        sf.write(fname, np.clip(pred, -1, 1).astype(np.float32), SAMPLE_RATE)
        print(f'Saved {fname}')
else:
    print('Install soundfile to save .wav files: pip install soundfile')

## Summary

| Architecture |  Key observation |
|---|---|
| **SIREN** |  Accurately fits fine-grained waveform structure |
| ReLU | Fits a rough average; cannot represent high frequencies |
| ReLU + PE | Slightly better than ReLU but still far behind SIREN |

This matches Table 3 of the paper (Bach MSE: SIREN ≈ 1.1×10⁻⁵ vs ReLU ≈ 2.5×10⁻²).
